In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

#  Create logs folder if not exists
os.makedirs("logs", exist_ok=True)

#  Logging Configuration (Fixed)
logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

#  Create SQLite Engine
engine = create_engine('sqlite:///inventory.db')

def ingest_db(df, table_name, conn, mode='replace'):
    """
    This function will ingest dataframe into database table safely
    """
    df.to_sql(
        table_name,
        con=engine,
        if_exists=mode,
        index=False,
        chunksize=500   #  FIX for SQLite 999 variable limit
    )

def load_raw_data():
    """
    This function loads CSV files and ingests them into DB safely
    """
    start = time.time()

    folder_path = r'C:\Users\Admin\Desktop\csvdata\\'  #  Change if needed

    for file in os.listdir(folder_path):
        if file.endswith('.csv'):

            logging.info(f"Processing started for {file}")
            print(f"\nProcessing: {file}")

            total_rows = 0
            total_columns = None

            file_path = os.path.join(folder_path, file)

            #  Read CSV in chunks (Large file safe)
            for i, chunk in enumerate(pd.read_csv(file_path, chunksize=20000)):

                total_rows += len(chunk)

                if i == 0:
                    total_columns = len(chunk.columns)
                    ingest_db(chunk, file[:-4], engine, 'replace')
                else:
                    ingest_db(chunk, file[:-4], engine, 'append')

            logging.info(f"{file} shape: ({total_rows}, {total_columns})")
            logging.info(f"{file} uploaded successfully")

            print(f"{file} shape: ({total_rows}, {total_columns})")
            print(f"{file} uploaded successfully ")

    end = time.time()
    total_time = (end - start) / 60

    logging.info("------------ Ingestion Complete ------------")
    logging.info(f"Total Time Taken: {total_time:.2f} minutes")

    print("\nAll files uploaded successfully ")
    print(f"Total Time Taken: {total_time:.2f} minutes")

if __name__ == '__main__':
    load_raw_data()


Processing: begin_inventory.csv
begin_inventory.csv shape: (206529, 9)
begin_inventory.csv uploaded successfully 

Processing: end_inventory.csv
end_inventory.csv shape: (224489, 9)
end_inventory.csv uploaded successfully 

Processing: purchases.csv
purchases.csv shape: (2372474, 16)
purchases.csv uploaded successfully 

Processing: purchase_prices.csv
purchase_prices.csv shape: (12261, 9)
purchase_prices.csv uploaded successfully 

Processing: sales.csv
sales.csv shape: (12825363, 14)
sales.csv uploaded successfully 

Processing: vendor_invoice.csv
vendor_invoice.csv shape: (5543, 10)
vendor_invoice.csv uploaded successfully 

All files uploaded successfully 
Total Time Taken: 11.61 minutes
